# La classe de langue : lisibilité, vocabulaire, textes à trous et quiz

**Cours** : Intelligence artificielle · Master 2 (Didactique, anglais) · Université Yahia Farès, Médéa
**Chapitre 2** : Lire et traiter des fichiers textes, appliqué à la classe de langue
**Durée** : environ 1 h 30

### Ce que vous allez faire
- Mesurer la **lisibilité** d'un texte avec une formule (Kandel et Moles pour le français, Flesch pour l'anglais), et voir ce que cette formule ne voit pas.
- Dresser le **profil lexical** d'un texte : occurrences, formes, rapport formes/occurrences, mots fréquents, hapax.
- Fabriquer des exercices : un **texte à trous** avec son corrigé, une **comparaison** entre la réponse d'un apprenant et une réponse modèle, un petit **quiz** à choix multiples.

### Mode d'emploi
Exécutez chaque cellule de haut en bas : cliquez sur le bouton ▶ à sa gauche, ou appuyez sur Maj + Entrée. **Vous n'avez jamais à écrire de code.** Modifiez seulement les champs des cellules intitulées « À MODIFIER ».

- Dans Google Colab, le menu « Exécution → Tout exécuter » (*Runtime → Run all*) lance tout le carnet. Les champs « À MODIFIER » s'affichent comme des cases, des curseurs ou des listes à côté du code.
- Dans Jupyter, ces champs sont de simples commentaires `#@param` : changez seulement la valeur placée après le signe `=`.
- Après avoir modifié une valeur, relancez la cellule, puis les cellules qui la suivent.

### La méthode : prédire, exécuter, observer, modifier
Avant chaque cellule, une question vous demande de **prédire** le résultat : notez votre réponse. Puis **exécutez** la cellule. La rubrique « Ce que vous devez voir » donne le résultat attendu et une question pour **observer** et interpréter. Enfin, vous **modifiez** une valeur pour vérifier que vous avez compris.

### Les textes
Des extraits du domaine public : Perrault (1697), Maupassant (1884), Hugo (1862), et deux discours pour le mini-projet (Hugo 1849, Jaurès 1903). Le « texte simple » et les réponses d'apprenants sont **inventés** pour l'exemple (données fictives).

## 0. Préparer les outils
Cette cellule charge les outils dont le carnet a besoin : `re` pour découper un texte en mots et en phrases, `Counter` pour compter, `difflib` pour comparer deux textes, `random` pour les tirages du quiz, `pandas` pour les tableaux et `matplotlib` pour le graphique. Ils sont tous déjà installés dans Colab : rien à télécharger.

**Avant d'exécuter :** que va afficher cette cellule, à votre avis ? Une longue liste, un tableau, ou presque rien ?

In [ ]:
#@title Préparer les outils (à exécuter en premier)
import re                          # expressions régulières : découper en mots et en phrases
import random                      # tirages au hasard (quiz), reproductibles grâce à une graine
import difflib                     # comparer deux textes
import io                          # lire une chaîne de caractères comme un fichier
from collections import Counter    # compter les fréquences
import pandas as pd                # tableaux
import matplotlib.pyplot as plt    # graphiques
pd.set_option("display.width", 140)
print("Outils prêts.")

### Ce que vous devez voir
```
Outils prêts.
```
Les lignes `import` n'affichent rien ; seule la dernière ligne, un `print()`, écrit ce message.

**Observez :** un numéro est apparu à gauche de la cellule. Que se passerait-il, selon vous, si vous exécutiez une cellule plus bas sans avoir exécuté celle-ci ?

## 1. La lisibilité d'un texte
Un enseignant de langue choisit sans cesse des textes : celui-ci est-il à la portée de ma classe ? En 1948, Rudolf Flesch a proposé une réponse chiffrée pour l'anglais : un texte est plus difficile quand ses **phrases sont longues** et quand ses **mots sont longs**, mesurés en syllabes. En 1958, L. Kandel et A. Moles ont adapté sa formule au français. Ces formules ne lisent pas le texte : elles comptent.

La cellule suivante range les quatre textes du carnet dans un dictionnaire (chapitre 2) : la clé est le nom du texte, la valeur est le texte lui-même. Elle affiche ensuite le nombre de mots de chacun.

**Avant d'exécuter :** classez ces quatre textes du plus facile au plus difficile pour des apprenants de niveau B1 : texte simple, Perrault, Maupassant, Hugo. Notez votre classement : vous le comparerez à celui de la formule.

In [ ]:
#@title Les quatre textes du carnet (domaine public, sauf le texte simple, inventé)
textes = {
    "Texte simple (inventé)": """Léa a un petit chat. Il s'appelle Pompon. Il est noir et blanc.
Le matin, Pompon dort sur le lit. L'après-midi, il joue dans le jardin avec une balle rouge.
Le soir, Léa lui donne du lait. Pompon boit vite. Puis il vient sur les genoux de Léa.
Il ferme les yeux. Léa lit un livre. Le chat et la fille sont contents.""",
    "Perrault, Le Petit Chaperon rouge (1697)": """Il était une fois une petite fille de village, la plus jolie qu'on eût su voir ; sa mère en était folle, et sa mère-grand plus folle encore. Cette bonne femme lui fit faire un petit chaperon rouge, qui lui seyait si bien, que partout on l'appelait le Petit Chaperon rouge. Un jour sa mère, ayant cuit et fait des galettes, lui dit : « Va voir comme se porte ta mère-grand, car on m'a dit qu'elle était malade. Porte-lui une galette et ce petit pot de beurre. » Le Petit Chaperon rouge partit aussitôt pour aller chez sa mère-grand, qui demeurait dans un autre village. En passant dans un bois, elle rencontra compère le Loup, qui eut bien envie de la manger ; mais il n'osa, à cause de quelques bûcherons qui étaient dans la forêt.""",
    "Maupassant, La Parure (1884)": """C'était une de ces jolies et charmantes filles, nées, comme par une erreur du destin, dans une famille d'employés. Elle n'avait pas de dot, pas d'espérances, aucun moyen d'être connue, comprise, aimée, épousée par un homme riche et distingué ; et elle se laissa marier avec un petit commis du ministère de l'Instruction publique. Elle fut simple ne pouvant être parée, mais malheureuse comme une déclassée ; car les femmes n'ont point de caste ni de race, leur beauté, leur grâce et leur charme leur servant de naissance et de famille. Leur finesse native, leur instinct d'élégance, leur souplesse d'esprit sont leur seule hiérarchie, et font des filles du peuple les égales des plus grandes dames. Elle souffrait sans cesse, se sentant née pour toutes les délicatesses et tous les luxes. Elle souffrait de la pauvreté de son logement, de la misère des murs, de l'usure des sièges, de la laideur des étoffes.""",
    "Hugo, Les Misérables (1862)": """En 1815, M. Charles-François-Bienvenu Myriel était évêque de Digne. C'était un vieillard d'environ soixante-quinze ans ; il occupait le siège de Digne depuis 1806. Quoique ce détail ne touche en aucune manière au fond même de ce que nous avons à raconter, il n'est peut-être pas inutile, ne fût-ce que pour être exact en tout, d'indiquer ici les bruits et les propos qui avaient couru sur son compte au moment où il était arrivé dans le diocèse. Vrai ou faux, ce qu'on dit des hommes tient souvent autant de place dans leur vie et surtout dans leur destinée que ce qu'ils font.""",
}
langue_des_textes = "fr"
for nom, t in textes.items():
    print(nom, ":", len(re.findall(r"\w[\w'’-]*", t)), "mots")

### Ce que vous devez voir
```
Texte simple (inventé) : 63 mots
Perrault, Le Petit Chaperon rouge (1697) : 132 mots
Maupassant, La Parure (1884) : 151 mots
Hugo, Les Misérables (1862) : 101 mots
```
Ici, un « mot » est une suite de lettres qui peut contenir une apostrophe ou un trait d'union : « C'était » et « mère-grand » comptent chacun pour un seul mot.

**Observez :** l'extrait de Hugo est le plus court. Est-ce une raison pour qu'il soit le plus facile ?

### À MODIFIER : le texte à étudier
Cette cellule est un **formulaire**. Dans Colab, `choix` est une liste déroulante qui propose les quatre textes ; `mon_texte` est une case où vous pouvez coller votre propre texte (un extrait de manuel, une production d'apprenant anonyme) ; `langue` indique la langue du texte, `fr` ou `en`. Si `mon_texte` n'est pas vide, c'est lui qui est étudié. Toute la suite du carnet travaille sur ce texte.

**Avant d'exécuter :** le texte choisi par défaut est celui de Maupassant. Par quels mots l'affichage commencera-t-il ?

In [ ]:
#@title À MODIFIER : le texte à étudier
choix = "Maupassant, La Parure (1884)" #@param ["Texte simple (inventé)", "Perrault, Le Petit Chaperon rouge (1697)", "Maupassant, La Parure (1884)", "Hugo, Les Misérables (1862)"]
mon_texte = "" #@param {type:"string"}
langue = "fr" #@param ["fr", "en"]
if mon_texte.strip() != "":
    texte = mon_texte              # votre propre texte, s'il y en a un
else:
    texte = textes[choix]          # sinon, le texte choisi dans la liste
print("Langue :", langue)
print("Début du texte :", texte[:160], "...")

### Ce que vous devez voir
```
Langue : fr
Début du texte : C'était une de ces jolies et charmantes filles, nées, comme par une erreur du destin, dans une famille d'employés. Elle n'avait pas de dot, pas d'espérances, au ...
```
`texte[:160]` garde les 160 premiers caractères : l'affichage s'arrête au milieu d'un mot.

**Observez :** si vous changez `choix` plus tard, quelles cellules faudra-t-il relancer pour que les résultats portent sur le nouveau texte ?

### Compter les syllabes, sans dictionnaire
Pour appliquer la formule, il faut compter les syllabes de chaque mot. Le programme n'a pas de dictionnaire de prononciation : il suit une **règle simple et transparente**.
1. Compter les **groupes de voyelles** qui se suivent : dans « oiseau », « oi » et « eau » font deux groupes, donc deux syllabes.
2. Retirer un **« e » final muet** quand il suit une consonne : « rivière » a trois groupes (i, iè, e) mais deux syllabes. En français, « -es » final est traité comme « -e » (« charmantes »).
3. En anglais, garder le « le » final après une consonne, qui se prononce (*lit-tle*, *ta-ble*).
4. Compter au moins une syllabe par mot.

La dernière ligne de la cellule applique cette règle aux huit premiers mots du texte.

**Avant d'exécuter :** comptez vous-même les syllabes de « C'était une de ces jolies et charmantes filles », à l'oral, comme en prose. Le programme trouvera-t-il la même chose que vous ?

In [ ]:
#@title Compter les syllabes d'un mot (outil)
VOYELLES = "aeiouyàâäéèêëîïôöùûüÿœæ"
def compter_syllabes(mot, langue):
    """Estimation : nombre de groupes de voyelles, moins un « e » final muet."""
    mot = mot.lower()
    n = len(re.findall(f"[{VOYELLES}]+", mot))     # « oi », « eau », « iè » : un seul groupe
    if langue == "fr" and mot.endswith("es"):
        mot = mot[:-1]                               # « charmantes » se traite comme « charmante »
    muet = len(mot) > 1 and mot.endswith("e") and mot[-2] not in VOYELLES
    if langue == "en" and mot.endswith("le") and len(mot) > 2 and mot[-3] not in VOYELLES:
        muet = False                                 # little, table : ce « le » final se prononce
    if muet:
        n = n - 1                                    # rivière, prince : le « e » final est muet
    return max(n, 1)                                 # au moins une syllabe par mot
print([(m, compter_syllabes(m, langue)) for m in re.findall(r"\w[\w'’-]*", texte)[:8]])

### Ce que vous devez voir
```
[("C'était", 2), ('une', 1), ('de', 1), ('ces', 1), ('jolies', 2), ('et', 1), ('charmantes', 2), ('filles', 1)]
```
Sur ces huit mots, la règle tombe juste. « une » (deux groupes, u et e) perd son « e » muet ; « jolies » garde ses deux syllabes, car son « e » suit une voyelle.

**Observez :** en poésie, le « e » muet se prononce devant une consonne : « char-man-tes fil-les ». Pourquoi ce programme ne conviendrait-il pas pour compter les syllabes d'un alexandrin ?

### Vérifier l'outil
Une règle simple se trompe forcément. Pour savoir où, on la compare à des réponses connues : le dictionnaire `reference` donne, pour onze mots, le nombre réel de syllabes, compté à la main (prononciation courante, en prose).

**Avant d'exécuter :** parmi ces onze mots (rivière, jolies, charmantes, était, oiseau, des, employés, poète, pays, chantent, 1815), lesquels vont piéger la règle ? Combien de réponses justes prévoyez-vous ?

In [ ]:
#@title Vérifier l'outil sur des mots dont on connaît la réponse
# Nombre réel de syllabes, à l'oral et en prose, compté à la main
reference = {"rivière": 2, "jolies": 2, "charmantes": 2, "était": 2, "oiseau": 2, "des": 1,
             "employés": 3, "poète": 2, "pays": 2, "chantent": 1, "1815": 4}
justes = 0
for mot, vrai in reference.items():
    estime = compter_syllabes(mot, "fr")
    if estime == vrai:
        justes = justes + 1
        print(f"{mot:<10} estimé {estime}, réel {vrai}   juste")
    else:
        print(f"{mot:<10} estimé {estime}, réel {vrai}   ERREUR")
print(justes, "justes sur", len(reference))

### Ce que vous devez voir
Six justes sur onze (`6 justes sur 11`). Les erreurs :
- « employés » (2 au lieu de 3) : « oyé » forme un seul groupe de voyelles, alors qu'on prononce *em-plo-yés* ;
- « poète » et « pays » (1 au lieu de 2) : deux voyelles qui se suivent peuvent appartenir à deux syllabes (*po-ète*, *pa-ys*) ;
- « chantent » (2 au lieu de 1) : la terminaison muette « -ent » des verbes n'est pas reconnue, et une règle d'orthographe ne peut pas distinguer *ils chantent* de *le moment* ;
- « 1815 » (1 au lieu de 4) : un nombre ne contient aucune voyelle, alors qu'on dit *mille huit cent quinze*.

**Observez :** ces onze mots ont été choisis pour piéger la règle ; ils ne disent pas son taux d'erreur sur un vrai texte (sur les huit premiers mots de Maupassant, elle ne s'est pas trompée). Comment estimeriez-vous honnêtement sa fiabilité ?

### La formule
La cellule définit la fonction `lisibilite(texte, langue)`. Elle découpe le texte en **phrases** (aux signes `.` `!` `?` `…`), puis en **mots**, compte les syllabes et calcule deux moyennes : le nombre de mots par phrase et le nombre de syllabes par mot. Le score vient ensuite :

- pour le français, **Kandel et Moles (1958)** : score = 206,835 − 1,015 × (mots par phrase) − 73,6 × (syllabes par mot) ; la constante est souvent arrondie à 207 ;
- pour l'anglais, **Flesch (1948)** : score = 206,835 − 1,015 × (mots par phrase) − 84,6 × (syllabes par mot).

Plus le score est **élevé**, plus le texte est **facile**. Les deux moyennes sont arrondies avant le calcul, pour que vous puissiez refaire l'opération à la calculatrice.

**Avant d'exécuter :** les phrases de Maupassant sont-elles plutôt courtes (moins de 12 mots) ou plutôt longues (plus de 20) ? Son score sera-t-il au-dessus ou au-dessous de 50 ?

In [ ]:
#@title La formule de lisibilité (outil) et le résultat pour votre texte
def lisibilite(texte, langue):
    phrases = [p for p in re.split(r"[.!?…]+", texte) if re.search(r"\w", p)]  # coupe aux . ! ? …
    mots = re.findall(r"\w[\w'’-]*", texte)            # « C'était », « mère-grand » : un seul mot
    syllabes = sum([compter_syllabes(m, langue) for m in mots])
    mpp = round(len(mots) / len(phrases), 1)            # longueur moyenne des phrases, en mots
    spm = round(syllabes / len(mots), 2)                # longueur moyenne des mots, en syllabes
    if langue == "en":
        score = 206.835 - 1.015 * mpp - 84.6 * spm      # Flesch (1948), pour l'anglais
    else:
        score = 206.835 - 1.015 * mpp - 73.6 * spm      # Kandel et Moles (1958), pour le français
    return {"phrases": len(phrases), "mots": len(mots), "syllabes": syllabes,
            "mots par phrase": mpp, "syllabes par mot": spm, "score": round(score, 1)}
for cle, valeur in lisibilite(texte, langue).items():
    print(cle, ":", valeur)

### Ce que vous devez voir
```
phrases : 6
mots : 151
syllabes : 214
mots par phrase : 25.2
syllabes par mot : 1.42
score : 76.7
```
À la calculatrice : 206,835 − 1,015 × 25,2 − 73,6 × 1,42 = 206,835 − 25,578 − 104,512 = 76,7 (arrondi). Python écrit les décimales avec un point : 25.2 se lit 25,2.

**Observez :** les phrases de Maupassant sont longues (25 mots en moyenne), mais ses mots sont courts. Lequel des deux termes de la formule enlève le plus de points au score ?

### Interpréter et comparer
Flesch a associé ses scores à des niveaux, calibrés sur des élèves américains de langue maternelle anglaise. On reprend souvent la même échelle pour le français, faute de mieux :

| Score | Niveau | Public visé par Flesch (école américaine) |
|---|---|---|
| 90 et plus | très facile | 10-11 ans |
| 80 à 90 | facile | 11-12 ans |
| 70 à 80 | assez facile | 12-13 ans |
| 60 à 70 | standard | 13-15 ans |
| 50 à 60 | assez difficile | 15-18 ans |
| 30 à 50 | difficile | études supérieures |
| moins de 30 | très difficile | diplômés de l'université |

Ces bandes sont **approximatives** : elles ne parlent ni d'apprenants de langue étrangère, ni des niveaux A1 à C2 du Cadre européen commun de référence pour les langues. La cellule définit `interpreter(score)`, qui traduit un score en niveau, puis calcule la lisibilité des quatre textes dans un tableau `pandas`, trié du plus facile au plus difficile.

**Avant d'exécuter :** reprenez votre classement du début. La formule sera-t-elle d'accord avec vous ?

In [ ]:
#@title Interpréter les scores et comparer les quatre textes
BANDES = [(90, "très facile"), (80, "facile"), (70, "assez facile"), (60, "standard"),
          (50, "assez difficile"), (30, "difficile")]
def interpreter(score):
    """Traduit un score en niveau, selon l'échelle de Flesch (reprise pour le français)."""
    for seuil, niveau in BANDES:
        if score >= seuil:
            return niveau
    return "très difficile"
tableau = pd.DataFrame([lisibilite(t, langue_des_textes) for t in textes.values()], index=list(textes))
tableau["niveau"] = [interpreter(s) for s in tableau["score"]]
print(tableau.sort_values("score", ascending=False).to_string())

### Ce que vous devez voir
```
                                          phrases  mots  syllabes  mots par phrase  syllabes par mot  score        niveau
Texte simple (inventé)                         11    63        76              5.7              1.21  112.0   très facile
Perrault, Le Petit Chaperon rouge (1697)        6   132       184             22.0              1.39   82.2        facile
Maupassant, La Parure (1884)                    6   151       214             25.2              1.42   76.7  assez facile
Hugo, Les Misérables (1862)                     5   101       150             20.2              1.49   76.7  assez facile
```
Trois remarques :
- Le texte simple dépasse 100 : l'échelle n'a pas de plafond, 100 n'est pas une note maximale.
- Maupassant et Hugo obtiennent **exactement le même score**, 76,7, pour des raisons opposées : Maupassant a des phrases plus longues, Hugo des mots plus longs.
- Hugo compte 5 phrases au lieu de 4 : le point de « M. Charles-François-Bienvenu Myriel » est pris pour une fin de phrase. Sans cette erreur, ses phrases feraient 25,2 mots en moyenne et son score tomberait à 71,6.

**Observez :** « ne fût-ce que pour être exact en tout » est-il difficile pour un apprenant à cause de la longueur de la phrase, ou pour une autre raison que la formule ne voit pas ?

### Les scores en image
Un diagramme en barres horizontales, une barre par texte. Les fines lignes grises verticales marquent les limites entre les niveaux (30, 50, 60, 70, 80, 90).

**Avant d'exécuter :** quel texte aura la barre la plus longue ? Les barres de Maupassant et de Hugo seront-elles différentes ?

In [ ]:
#@title Les scores en image
ordre = tableau.sort_values("score")
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(ordre.index, ordre["score"], height=0.5, color="#2a78d6")
for y, s in enumerate(ordre["score"]):
    ax.text(s + 1.5, y, f"{s}  ({interpreter(s)})", va="center", fontsize=9)
for seuil, niveau in BANDES:
    ax.axvline(seuil, color="#dddddd", linewidth=1, zorder=0)    # limites entre les niveaux
ax.set_xlim(min(0, ordre["score"].min() - 5), 125)
ax.set_xlabel("Score de lisibilité (plus il est haut, plus le texte est facile)")
ax.set_title("Lisibilité des quatre textes (formule de Kandel et Moles)")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### Ce que vous devez voir
Quatre barres bleues, du score le plus élevé (en haut) au plus faible (en bas) : texte simple 112.0 (très facile), Perrault 82.2 (facile), puis Maupassant et Hugo, deux barres de même longueur à 76.7 (assez facile). Chaque barre porte son score et son niveau.

**Observez :** le graphique donne une impression de précision (une barre, un chiffre avec une décimale). Quelle légende écririez-vous dessous pour qu'un collègue ne le prenne pas pour une mesure exacte de la difficulté ?

### Pourquoi une formule différente pour le français ?
Les deux formules ne diffèrent que par un coefficient : 84,6 pour l'anglais, 73,6 pour le français. Les mots français comptent en moyenne plus de syllabes que les mots anglais ; avec le coefficient anglais, tous les textes français paraîtraient trop difficiles. Cette cellule applique les deux formules au texte choisi.

**Avant d'exécuter :** avec la formule anglaise, le score de Maupassant sera-t-il plus haut ou plus bas que 76,7 ? De combien, à peu près ?

In [ ]:
#@title Les deux formules sur le même texte
r = lisibilite(texte, langue)
flesch = 206.835 - 1.015 * r["mots par phrase"] - 84.6 * r["syllabes par mot"]
kandel_moles = 206.835 - 1.015 * r["mots par phrase"] - 73.6 * r["syllabes par mot"]
print("Formule de Flesch (anglais)           :", round(flesch, 1), "->", interpreter(flesch))
print("Formule de Kandel et Moles (français) :", round(kandel_moles, 1), "->", interpreter(kandel_moles))
print("Écart :", round(kandel_moles - flesch, 1), "points, soit 11 x", r["syllabes par mot"], "syllabes par mot")

### Ce que vous devez voir
```
Formule de Flesch (anglais)           : 61.1 -> standard
Formule de Kandel et Moles (français) : 76.7 -> assez facile
Écart : 15.6 points, soit 11 x 1.42 syllabes par mot
```
Le même texte passe de « standard » à « assez facile » selon la formule. L'écart vaut exactement (84,6 − 73,6) × 1,42 = 11 × 1,42 = 15,6 points.

**Observez :** un score n'a de sens que comparé à d'autres scores obtenus avec la même formule, dans la même langue. Peut-on dire qu'un texte anglais à 70 est « plus facile » qu'un texte français à 65 ?

## 2. Le profil lexical d'un texte
Comme au chapitre 2, on distingue les **occurrences** (*tokens*), chaque apparition d'un mot, et les **formes** (*types*), les mots différents. Leur rapport, le **TTR** (*type/token ratio*), mesure grossièrement la variété du vocabulaire : proche de 1, presque chaque mot est nouveau ; plus bas, le texte répète davantage les mêmes mots.

Le découpage est celui du chapitre 2, `re.findall(r"\w+", …)` sur le texte en minuscules : « l'Instruction » donne deux occurrences, « l » et « instruction ». Il y a donc un peu plus d'occurrences que de mots dans la partie 1.

**Avant d'exécuter :** combien de formes différentes le texte de Maupassant contient-il, à peu près ? Plus ou moins de la moitié de ses occurrences ?

In [ ]:
#@title Occurrences, formes et richesse du vocabulaire
tokens = re.findall(r"\w+", texte.lower())      # comme au chapitre 2 : « l'herbe » donne l + herbe
types = set(tokens)                              # chaque forme n'est gardée qu'une fois
print("Occurrences (tokens) :", len(tokens))
print("Formes différentes (types) :", len(types))
print("Rapport formes/occurrences (TTR) :", round(len(types) / len(tokens), 2))

### Ce que vous devez voir
```
Occurrences (tokens) : 161
Formes différentes (types) : 104
Rapport formes/occurrences (TTR) : 0.65
```
161 occurrences, et non 151 : les élisions (« d'employés », « l'usure ») sont coupées en deux. 104 formes pour 161 occurrences : près des deux tiers.

**Observez :** quels mots, selon vous, reviennent le plus souvent dans ce passage ? Notez trois hypothèses avant la cellule sur les fréquences.

### Attention : le TTR dépend de la longueur
Plus un texte est long, plus il réutilise des mots déjà vus (articles, pronoms, prépositions) : son TTR baisse mécaniquement. Pour comparer deux textes, il faut donc les comparer **à longueur égale**, par exemple sur leurs 50 premiers mots. La cellule calcule les deux TTR pour chacun des quatre textes.

**Avant d'exécuter :** le texte simple, qui répète « Léa » et « Pompon », aura-t-il le TTR le plus bas des quatre ?

In [ ]:
#@title Le TTR dépend de la longueur du texte
lignes = []
for nom, t in textes.items():
    toks = re.findall(r"\w+", t.lower())
    lignes.append({"texte": nom, "tokens": len(toks),
                   "TTR (texte entier)": round(len(set(toks)) / len(toks), 2),
                   "TTR (50 premiers mots)": round(len(set(toks[:50])) / 50, 2)})
print(pd.DataFrame(lignes).to_string(index=False))

### Ce que vous devez voir
```
                                   texte  tokens  TTR (texte entier)  TTR (50 premiers mots)
                  Texte simple (inventé)      66                0.71                    0.80
Perrault, Le Petit Chaperon rouge (1697)     141                0.62                    0.84
            Maupassant, La Parure (1884)     161                0.65                    0.80
             Hugo, Les Misérables (1862)     112                0.76                    0.86
```
Sur les textes entiers, le texte simple (0,71) paraît plus varié que Perrault (0,62) et Maupassant (0,65) : c'est un effet de sa brièveté (66 occurrences). Sur les 50 premiers mots, il n'est plus devant personne : il est à égalité avec Maupassant (0,80), derrière Perrault (0,84) et Hugo (0,86).

**Observez :** pour comparer la richesse lexicale de deux copies d'apprenants, l'une de 80 mots, l'autre de 300, que feriez-vous avant de calculer leur TTR ?

### Les mots les plus fréquents
`Counter` compte chaque forme. Sans filtre, les premières places reviennent aux **mots vides** (*stop words*) : articles, prépositions, pronoms, auxiliaires. La cellule définit une liste de mots vides, puis refait le comptage sans eux. Cette liste servira aussi au texte à trous et au quiz.

**Avant d'exécuter :** reprenez vos trois hypothèses. Quel est, selon vous, le mot le plus fréquent du passage, mots vides compris ?

In [ ]:
#@title Les mots les plus fréquents, avec et sans les mots vides
mots_vides = {"le", "la", "les", "l", "un", "une", "des", "de", "du", "d", "au", "aux", "et", "ou",
              "mais", "ni", "car", "que", "qu", "qui", "où", "à", "en", "dans", "sur", "sous", "par",
              "pour", "avec", "sans", "ce", "c", "ces", "cet", "cette", "il", "ils", "elle", "elles",
              "on", "je", "j", "tu", "nous", "vous", "se", "s", "ne", "n", "pas", "point", "lui", "leur",
              "leurs", "son", "sa", "ses", "mon", "ma", "mes", "m", "t", "y", "est", "était", "sont",
              "fut", "être", "a", "avait", "ont", "eut", "comme", "si", "plus", "très", "tout", "tous"}
freq = Counter(tokens)
print("Avec les mots vides :", freq.most_common(6))
pleins = [m for m in tokens if m not in mots_vides]
print("Sans les mots vides :", Counter(pleins).most_common(6))

### Ce que vous devez voir
```
Avec les mots vides : [('de', 12), ('leur', 8), ('et', 7), ('d', 5), ('elle', 5), ('des', 5)]
Sans les mots vides : [('filles', 2), ('famille', 2), ('souffrait', 2), ('jolies', 1), ('charmantes', 1), ('nées', 1)]
```
Sans les mots vides, aucun mot ne dépasse deux occurrences ; à égalité, `most_common` garde l'ordre d'apparition dans le texte. Mais « leur », un mot vide, apparaît 8 fois en deux phrases : « leur beauté, leur grâce et leur charme… ».

**Observez :** « leur » répété huit fois, « Elle souffrait » deux fois en tête de phrase : quel procédé de style le comptage révèle-t-il ? Le programme le « comprend »-il ?

### Les hapax
Un **hapax** est un mot qui n'apparaît qu'une seule fois dans un texte. Pour l'enseignant, les hapax longs sont des candidats pour le vocabulaire à expliquer avant la lecture : l'apprenant ne pourra pas s'appuyer sur une deuxième rencontre du mot pour en deviner le sens.

**Avant d'exécuter :** sur les 104 formes du passage, combien sont des hapax, selon vous : un quart, la moitié, les trois quarts ?

In [ ]:
#@title Les mots employés une seule fois (hapax)
hapax = [m for m, n in freq.items() if n == 1]
print(len(hapax), "hapax sur", len(types), "formes, soit", round(100 * len(hapax) / len(types)), "%")
# Des mots peut-être à expliquer avant la lecture : les hapax pleins et longs
a_expliquer = sorted([m for m in hapax if m not in mots_vides and len(m) >= 8])
print("Hapax de 8 lettres ou plus :", a_expliquer)

### Ce que vous devez voir
```
83 hapax sur 104 formes, soit 80 %
Hapax de 8 lettres ou plus : ['charmantes', 'comprise', 'distingué', 'déclassée', 'délicatesses', 'employés', 'espérances', 'hiérarchie', 'instinct', 'instruction', 'logement', 'malheureuse', 'ministère', 'naissance', 'pauvreté', 'publique', 'souplesse', 'élégance']
```
Quatre formes sur cinq n'apparaissent qu'une fois : c'est habituel dans un texte court. `sorted()` range « élégance » à la fin : « é » est classé après « z », le même piège qu'au chapitre 2.

**Observez :** parmi ces 18 mots, lesquels expliqueriez-vous vraiment à des apprenants de niveau B1 ? Lesquels sont transparents (« instinct », « hiérarchie ») ou faciles à deviner grâce au contexte ?

## 3. Un texte à trous automatique
Le **test de closure** (*cloze test*), proposé par Wilson Taylor en 1953, consiste à effacer des mots d'un texte et à demander au lecteur de les retrouver. Taylor l'avait d'abord conçu… comme une mesure de la lisibilité. Deux manières classiques de choisir les mots :
- effacer **un mot sur n** (ici, un mot *plein* sur n : on saute les mots vides) : l'exercice porte plutôt sur la compréhension globale ;
- effacer les **mots longs** : l'exercice porte plutôt sur le vocabulaire.

La fonction `a_effacer(mot, rang, mode, n, longueur_min)` applique la règle à un mot ; `rang` est la position du mot parmi les mots pleins. La cellule l'essaie sur les huit premiers mots pleins du texte, avec n = 3 et une longueur minimale de 7 lettres.

**Avant d'exécuter :** parmi « jolies, charmantes, filles, nées, erreur, destin, famille, employés », lesquels seront effacés avec la règle « un sur 3 » ? avec la règle « 7 lettres ou plus » ?

In [ ]:
#@title La règle : quels mots effacer ? (outil)
def a_effacer(mot, rang, mode, n, longueur_min):
    """Vrai si ce mot plein doit devenir un trou."""
    if mode == "un mot plein sur n":
        return rang % n == 0                  # le n-ième mot plein, le 2n-ième, etc.
    else:
        return len(mot) >= longueur_min       # tous les mots assez longs
# Essai sur les huit premiers mots pleins du texte (n = 3, longueur minimale = 7)
pleins_texte = [m for m in re.findall(r"\w+", texte) if m.isalpha() and m.lower() not in mots_vides]
for rang, mot in enumerate(pleins_texte[:8], start=1):
    un_sur_3 = a_effacer(mot, rang, "un mot plein sur n", 3, 7)
    longs = a_effacer(mot, rang, "mots longs", 3, 7)
    print(f"{rang}. {mot:<12}", "un sur 3 :", un_sur_3, "|", "7 lettres ou plus :", longs)

### Ce que vous devez voir
```
1. jolies       un sur 3 : False | 7 lettres ou plus : False
2. charmantes   un sur 3 : False | 7 lettres ou plus : True
3. filles       un sur 3 : True | 7 lettres ou plus : False
4. nées         un sur 3 : False | 7 lettres ou plus : False
5. erreur       un sur 3 : False | 7 lettres ou plus : False
6. destin       un sur 3 : True | 7 lettres ou plus : False
7. famille      un sur 3 : False | 7 lettres ou plus : True
8. employés     un sur 3 : False | 7 lettres ou plus : True
```
`True` veut dire « à effacer ». « Un sur 3 » efface le 3e et le 6e mot plein (filles, destin) : `rang % n == 0` vérifie que le reste de la division de `rang` par `n` est nul. « 7 lettres ou plus » efface charmantes, famille et employés.

**Observez :** la règle « un sur 3 » ne regarde pas le mot. Est-ce un défaut ou une qualité pour un exercice de compréhension ?

### Fabriquer le texte à trous
La fonction `texte_a_trous` parcourt le texte morceau par morceau : `re.findall(r"\w+|\W+", texte)` le découpe en mots et en « non-mots » (espaces, ponctuation), ce qui permet de le recomposer à l'identique. Chaque mot plein à effacer est remplacé par un trou numéroté et rangé dans le corrigé. L'option `initiale` laisse la première lettre comme indice. La dernière ligne essaie la fonction sur un vers de Rimbaud (chapitre 2), en effaçant les mots de 6 lettres ou plus.

**Avant d'exécuter :** dans « Il dort dans le soleil, la main sur sa poitrine. », quels mots ont 6 lettres ou plus ?

In [ ]:
#@title Fabriquer un texte à trous (outil)
def texte_a_trous(texte, mode, n, longueur_min, initiale):
    resultat = ""                                    # le texte à trous, construit morceau par morceau
    corrige = []                                     # les mots effacés, dans l'ordre
    rang = 0                                         # numéro du mot plein en cours
    for morceau in re.findall(r"\w+|\W+", texte):    # des mots et des « non-mots » (espaces, ponctuation)
        if morceau.isalpha() and morceau.lower() not in mots_vides:
            rang = rang + 1
            if a_effacer(morceau, rang, mode, n, longueur_min):
                corrige.append(morceau)
                indice = morceau[0] if initiale else ""      # première lettre, si l'option est cochée
                morceau = f"({len(corrige)}) {indice}________"  # le mot devient un trou
        resultat = resultat + morceau
    return resultat, corrige
print(texte_a_trous("Il dort dans le soleil, la main sur sa poitrine.", "mots longs", 3, 6, False))

### Ce que vous devez voir
```
('Il dort dans le (1) ________, la main sur sa (2) ________.', ['soleil', 'poitrine'])
```
La fonction renvoie deux choses à la fois (un *tuple*, chapitre 2) : le texte à trous et la liste des mots effacés. La ponctuation est conservée.

**Observez :** pour cet exercice, le verbe « dort » aurait-il été un meilleur trou que « soleil » ? Que perd-on quand la machine choisit les mots d'après leur longueur ?

### À MODIFIER : réglages du texte à trous
Choisissez le `mode` (« un mot plein sur n » ou « mots longs »), la valeur de `n`, la longueur minimale `longueur_min` (utilisée seulement en mode « mots longs ») et l'option `initiale`. La cellule affiche le texte à trous, le corrigé, une **banque de mots** en ordre alphabétique (pour une version plus facile, en pédagogie différenciée) et le nombre de trous.

**Avant d'exécuter :** avec un mot plein sur 4, dans un passage de 161 occurrences dont beaucoup de mots vides, combien de trous prévoyez-vous ?

In [ ]:
#@title À MODIFIER : réglages du texte à trous
mode = "un mot plein sur n" #@param ["un mot plein sur n", "mots longs"]
n = 4 #@param {type:"slider", min:2, max:10, step:1}
longueur_min = 9 #@param {type:"slider", min:5, max:12, step:1}
initiale = False #@param {type:"boolean"}
trous, corrige = texte_a_trous(texte, mode, n, longueur_min, initiale)
print(trous)
print()
print("Corrigé :", ", ".join([f"({i}) {mot}" for i, mot in enumerate(corrige, start=1)]))
print("Banque de mots (ordre alphabétique) :", " - ".join(sorted(corrige, key=str.lower)))
print(len(corrige), "trous pour", len(re.findall(r"\w+", texte)), "mots")

### Ce que vous devez voir
Le passage de Maupassant avec 17 trous ; il commence ainsi : « C'était une de ces jolies et charmantes filles, (1) ________, comme par une erreur du destin, dans une famille d'(2) ________. » Puis :
```
Corrigé : (1) nées, (2) employés, (3) moyen, (4) épousée, (5) laissa, (6) ministère, (7) pouvant, (8) femmes, (9) grâce, (10) famille, (11) élégance, (12) hiérarchie, (13) égales, (14) cesse, (15) délicatesses, (16) logement, (17) sièges
Banque de mots (ordre alphabétique) : cesse - délicatesses - employés - famille - femmes - grâce - hiérarchie - laissa - logement - ministère - moyen - nées - pouvant - sièges - égales - élégance - épousée
17 trous pour 161 mots
```
**Modifiez :** passez en mode « mots longs » (avec `longueur_min` = 9), puis cochez `initiale`.

**Observez :** quelle version donneriez-vous à des débutants, laquelle à des avancés ? Quels trous corrigeriez-vous à la main : un trou impossible à deviner, un trou qui accepte plusieurs bonnes réponses (« moyen », « pouvant ») ?

## 4. Comparer la réponse d'un apprenant à une réponse modèle
Le module `difflib` compare deux suites et calcule un **taux de similarité** entre 0 (rien en commun) et 1 (identiques) : deux fois le nombre d'éléments communs, divisé par le nombre total d'éléments des deux suites. Ici, les éléments sont des **mots** ; la ponctuation reste collée au mot. `difflib` décrit le passage du modèle à la réponse par quatre opérations : `equal` (passage identique), `delete` (mots supprimés), `insert` (mots ajoutés), `replace` (mots remplacés). La fonction `comparer` renvoie ce taux et la réponse avec les différences marquées : `[-mot-]` pour un mot du modèle absent, `{+mot+}` pour un mot écrit par l'apprenant.

**Avant d'exécuter :** « le chat dort sur le lit » et « le chien dort sur le lit » ont 5 mots communs, sur 12 mots au total. Quel taux de similarité prévoyez-vous ?

In [ ]:
#@title Comparer deux réponses mot à mot (outil)
def comparer(modele, reponse):
    a = modele.split()                                  # on compare des listes de mots
    b = reponse.split()
    comparaison = difflib.SequenceMatcher(None, a, b)
    marque = []
    for operation, i1, i2, j1, j2 in comparaison.get_opcodes():
        if operation == "equal":                        # passage identique : on le recopie
            marque.append(" ".join(a[i1:i2]))
        if operation in ("delete", "replace"):          # mots du modèle absents de la réponse
            marque.append("[-" + " ".join(a[i1:i2]) + "-]")
        if operation in ("insert", "replace"):          # mots ajoutés ou changés par l'apprenant
            marque.append("{+" + " ".join(b[j1:j2]) + "+}")
    return round(comparaison.ratio(), 2), " ".join(marque)
print(comparer("le chat dort sur le lit", "le chien dort sur le lit"))

### Ce que vous devez voir
```
(0.83, 'le [-chat-] {+chien+} dort sur le lit')
```
Le calcul : 2 × 5 mots communs / 12 mots au total = 0,83. Un seul mot a changé, et c'est pourtant tout le sens de la phrase qui change.

**Observez :** d'après ce calcul, quel taux obtiendrait une réponse qui dit à peu près la même chose avec d'autres mots, comme « un félin sommeille sur la couche » ?

### À MODIFIER : la réponse d'un apprenant
Le formulaire contient une question de compréhension sur le passage de Maupassant, une réponse modèle et la réponse d'un apprenant. **Cette réponse est inventée** pour l'exemple : elle contient trois fautes typiques. Remplacez-la par une vraie réponse (anonyme) pour essayer l'outil.

**Avant d'exécuter :** repérez les fautes de l'apprenant dans le champ `reponse`. Le taux sera-t-il plus proche de 0,5 ou de 1 ?

In [ ]:
#@title À MODIFIER : la réponse d'un apprenant (réponse inventée)
question = "Pourquoi la jeune femme est-elle malheureuse ?" #@param {type:"string"}
modele = "Elle est malheureuse parce qu'elle est née dans une famille modeste et qu'elle a épousé un petit employé." #@param {type:"string"}
reponse = "Elle est malheureuse parce qu'elle est né dans une famille modeste et qu'elle a épousée un petit employer." #@param {type:"string"}
score, marque = comparer(modele, reponse)
print("Question :", question)
print("Similarité :", score, "soit", round(100 * score), "%")
print("Différences :", marque)
print("(Légende : [-mot-] = mot du modèle absent ; {+mot+} = mot écrit par l'apprenant)")

### Ce que vous devez voir
```
Question : Pourquoi la jeune femme est-elle malheureuse ?
Similarité : 0.83 soit 83 %
Différences : Elle est malheureuse parce qu'elle est [-née-] {+né+} dans une famille modeste et qu'elle a [-épousé-] {+épousée+} un petit [-employé.-] {+employer.+}
(Légende : [-mot-] = mot du modèle absent ; {+mot+} = mot écrit par l'apprenant)
```
Les trois fautes (accord oublié de « née », accord fautif de « épousé », confusion entre « employé » et « employer ») sont bien repérées. Mais « employé. » et « employer. » sont traités comme deux mots entièrement différents : l'outil ne voit pas qu'une seule lettre les sépare.

**Observez :** pour corriger une **dictée**, cet outil vous ferait-il gagner du temps ? Et pour noter une réponse de compréhension ?

### Similarité n'est pas justesse
Trois apprenants (**inventés**) ont répondu à la même question. Pour chacun, la cellule affiche le taux de similarité avec le modèle, les différences, et l'avis d'un enseignant (lui aussi inventé) sur la justesse de la réponse.

**Avant d'exécuter :** lisez les trois réponses dans le code. Classez-les de la meilleure à la moins bonne. Le taux de similarité donnera-t-il le même classement ?

In [ ]:
#@title Trois apprenants (réponses inventées) : similarité ou justesse ?
reponses = {
    "Apprenant A": ("Elle est malheureuse parce qu'elle est né dans une famille modeste et qu'elle a épousée un petit employer.",
                    "sens juste, trois fautes d'orthographe"),
    "Apprenant B": ("Sa famille n'avait pas d'argent et son mari n'est qu'un commis : elle se sent déclassée.",
                    "juste et bien reformulé"),
    "Apprenant C": ("Elle est heureuse parce qu'elle est née dans une famille riche et qu'elle a épousé un petit employé.",
                    "faux : contresens complet"),
}
for nom, (texte_reponse, avis) in reponses.items():
    score, marque = comparer(modele, texte_reponse)
    print(f"{nom} : similarité {score}   |   avis de l'enseignant (inventé) : {avis}")
    print("    ", marque)

### Ce que vous devez voir
```
Apprenant A : similarité 0.83   |   avis de l'enseignant (inventé) : sens juste, trois fautes d'orthographe
Apprenant B : similarité 0.12   |   avis de l'enseignant (inventé) : juste et bien reformulé
Apprenant C : similarité 0.89   |   avis de l'enseignant (inventé) : faux : contresens complet
```
(chaque ligne est suivie de la réponse avec ses différences marquées).

Le classement de la machine est l'inverse de celui de l'enseignant. C, qui fait un **contresens complet** (« heureuse », « riche »), obtient le meilleur taux, 0,89 ; B, qui a **compris et reformulé** avec ses propres mots, le plus mauvais, 0,12. Le taux mesure la ressemblance de surface avec le modèle, pas la justesse : **similarité n'est pas correction**. Il peut aider à préparer la correction d'une dictée ou à repérer des copies identiques ; il ne doit jamais noter seul une réponse ouverte.

**Observez :** quelle sorte de réponse ce taux favorise-t-il toujours ? Quel effet aurait sur les apprenants l'annonce qu'ils seront notés de cette façon ?

## 5. Un petit quiz de vocabulaire
Le quiz est un exercice à choix multiples : dans une phrase du texte, un mot est effacé, et l'apprenant le retrouve parmi quatre propositions. Cette première cellule prépare les ingrédients :
- les **mots candidats** : 5 lettres ou plus, et pas de mots vides ;
- les **phrases utilisables** : celles qui contiennent au moins un candidat ; dans chacune, le mot à effacer est le candidat **le plus long**.

**Avant d'exécuter :** dans la première phrase de Maupassant (« C'était une de ces jolies et charmantes filles, nées, comme par une erreur du destin, dans une famille d'employés »), quel mot sera effacé ?

In [ ]:
#@title Les ingrédients du quiz : mots candidats et phrases utilisables
# Mots candidats : 5 lettres ou plus, et pas des mots vides (triés, pour un tirage reproductible)
candidats = sorted(set([m for m in re.findall(r"\w+", texte) if len(m) >= 5 and m.lower() not in mots_vides]))
# Phrases utilisables : celles qui contiennent au moins un mot candidat
phrases_quiz = []
for p in re.split(r"[.!?…]+", texte):
    mots_p = [m for m in re.findall(r"\w+", p) if m in candidats]
    if mots_p:
        phrases_quiz.append((p.strip(), max(mots_p, key=len)))   # (la phrase, son mot le plus long)
print(len(candidats), "mots candidats, par exemple :", candidats[:6])
print(len(phrases_quiz), "phrases utilisables. La première, avec le mot à effacer :")
print(phrases_quiz[:1])

### Ce que vous devez voir
```
61 mots candidats, par exemple : ['Instruction', 'aimée', 'aucun', 'beauté', 'caste', 'cesse']
6 phrases utilisables. La première, avec le mot à effacer :
[("C'était une de ces jolies et charmantes filles, nées, comme par une erreur du destin, dans une famille d'employés", 'charmantes')]
```
« Instruction » vient en tête parce que Python range les majuscules avant les minuscules. La dernière ligne affiche une liste qui contient la première paire (phrase, mot à effacer) : « charmantes » (10 lettres) l'emporte sur « employés » (8).

**Observez :** le mot le plus long est-il toujours le plus intéressant à faire retrouver ? Proposez une autre règle.

### À MODIFIER : réglages du quiz
Choisissez le nombre de questions `nb_questions` et une **graine** (`graine`, *seed* en anglais). Le programme tire au hasard les phrases et les trois **distracteurs** (les mauvaises réponses), pris parmi les autres mots candidats du texte. `random.seed(graine)` fixe le point de départ du hasard : avec la même graine, on obtient toujours le même quiz, ce qui permet de le réimprimer ou de le partager ; avec une autre graine, on obtient un autre quiz.

**Avant d'exécuter :** les distracteurs sont tirés au hasard. Seront-ils plausibles (même nature grammaticale, même genre, même nombre que la bonne réponse) ?

In [ ]:
#@title À MODIFIER : réglages du quiz à choix multiples
nb_questions = 3 #@param {type:"slider", min:1, max:6, step:1}
graine = 2026 #@param {type:"integer"}
random.seed(graine)                                  # même graine = même quiz, à chaque exécution
tirage = random.sample(phrases_quiz, min(nb_questions, len(phrases_quiz)))
corrige_quiz = []
for numero, (phrase, cible) in enumerate(tirage, start=1):
    choix_quiz = random.sample([c for c in candidats if c != cible], 3) + [cible]
    random.shuffle(choix_quiz)                       # la bonne réponse change de place
    print(f"{numero}. {phrase.replace(cible, '________', 1)}")
    for lettre, mot in zip("ABCD", choix_quiz):
        print(f"     {lettre}) {mot}")
    bonne_lettre = "ABCD"[choix_quiz.index(cible)]
    corrige_quiz.append(f"{numero} : {bonne_lettre} ({cible})")
print("\nCorrigé :", " | ".join(corrige_quiz))

### Ce que vous devez voir
```
1. C'était une de ces jolies et ________ filles, nées, comme par une erreur du destin, dans une famille d'employés
     A) égales
     B) délicatesses
     C) élégance
     D) charmantes
2. Elle fut simple ne pouvant être parée, mais ________ comme une déclassée ; car les femmes n'ont point de caste ni de race, leur beauté, leur grâce et leur charme leur servant de naissance et de famille
     A) seule
     B) moyen
     C) servant
     D) malheureuse
3. Elle ________ de la pauvreté de son logement, de la misère des murs, de l'usure des sièges, de la laideur des étoffes
     A) souffrait
     B) cesse
     C) naissance
     D) charme

Corrigé : 1 : D (charmantes) | 2 : D (malheureuse) | 3 : A (souffrait)
```
Relancez la cellule : le quiz est identique, grâce à la graine. Beaucoup de distracteurs se trahissent par la grammaire : à la question 1, seul « égales » est, comme la réponse, un adjectif au féminin pluriel ; à la question 3, seul « souffrait » est un verbe. Un apprenant peut répondre juste sans connaître le mot.

**Modifiez :** changez `graine` (par exemple 7), puis `nb_questions`.

**Observez :** combien de ces questions garderiez-vous telles quelles pour une vraie classe ? Comment choisiriez-vous de meilleurs distracteurs ?

## À vous
Trois modifications à essayer avec les champs « À MODIFIER » (relancez ensuite les cellules qui suivent) :
1. Dans le choix du texte, prenez **Hugo**, puis relancez les parties 1 à 5. Retrouvez-vous l'erreur sur « M. » ? Quels mots le texte à trous et le quiz choisissent-ils ?
2. Collez dans `mon_texte` un paragraphe d'un manuel que vous utilisez (ou une production d'apprenant, anonyme) et comparez son score à ceux des quatre textes. Collez ensuite un texte anglais en choisissant `langue = "en"` : c'est alors la formule de Flesch qui s'applique (la liste de mots vides, elle, reste celle du français).
3. Préparez deux versions du même texte à trous : une pour débutants (« mots longs », `initiale` cochée, avec la banque de mots) et une pour avancés (« un mot plein sur n » avec n = 5, sans banque). Qu'est-ce qui rend l'une plus facile que l'autre ?

## Pour le mini-projet
Ces outils s'appliquent directement au **sujet 3** (discours politiques) : comparer la lisibilité et la richesse lexicale de discours de plusieurs orateurs, époques ou langues. Ils servent aussi au **sujet 1**, pour décrire la longueur et le vocabulaire des commentaires de lecteurs.

Préparez un fichier CSV (dans un tableur : « Enregistrer sous », format CSV, encodage UTF-8) avec deux colonnes, `titre` et `texte`, un texte par ligne. Dans Colab, cochez `charger` puis exécutez la cellule : une fenêtre vous demande de choisir le fichier (`files.upload()`), que `pandas.read_csv` transforme en tableau. Hors de Colab, cette fonction n'existe pas : le bloc `try`/`except` affiche alors un message au lieu d'une erreur. Sans fichier, la cellule utilise deux extraits de discours du domaine public (Hugo 1849, Jaurès 1903).

**Avant d'exécuter :** `charger` n'est pas coché. Quel tableau la cellule va-t-elle utiliser ?

In [ ]:
#@title (Facultatif) Vos propres textes : un fichier CSV avec les colonnes « titre » et « texte »
charger = False #@param {type:"boolean"}
exemple = """titre,texte
"Hugo, Discours sur la misère (1849)","Je ne suis pas, messieurs, de ceux qui croient qu'on peut supprimer la souffrance en ce monde ; la souffrance est une loi divine ; mais je suis de ceux qui pensent et qui affirment qu'on peut détruire la misère. Remarquez-le bien, messieurs, je ne dis pas diminuer, amoindrir, limiter, circonscrire, je dis détruire."
"Jaurès, Discours à la jeunesse (1903)","Le courage, c'est d'aimer la vie et de regarder la mort d'un regard tranquille ; c'est d'aller à l'idéal et de comprendre le réel ; c'est d'agir et de se donner aux grandes causes sans savoir quelle récompense réserve à notre effort l'univers profond, ni s'il lui réserve une récompense."
"""
corpus = pd.read_csv(io.StringIO(exemple))          # par défaut : les deux extraits ci-dessus
if charger:
    try:
        from google.colab import files
        fichiers = files.upload()                   # une fenêtre s'ouvre : choisissez votre fichier .csv
        corpus = pd.read_csv(list(fichiers)[0])
    except ImportError:
        print("Hors de Colab : placez le fichier à côté du carnet et écrivez corpus = pd.read_csv('mon_fichier.csv')")
print(corpus["titre"].tolist())

### Ce que vous devez voir
```
['Hugo, Discours sur la misère (1849)', 'Jaurès, Discours à la jeunesse (1903)']
```
Les titres des deux extraits intégrés au carnet. Avec votre propre fichier, ce sont vos titres qui s'affichent.

**Observez :** l'extrait de Hugo contient des virgules. Pourquoi faut-il l'entourer de guillemets dans le fichier CSV ?

La cellule applique à chaque texte du fichier la formule de lisibilité et le calcul du TTR, et rassemble les résultats dans un tableau. Elle utilise la `langue` choisie dans la partie 1.

**Avant d'exécuter :** l'extrait de Jaurès est une seule longue phrase, construite sur la répétition de « c'est ». Sera-t-il jugé plus facile ou plus difficile que celui de Hugo ?

In [ ]:
#@title Lisibilité et vocabulaire de chaque texte du fichier
lignes = []
for titre, contenu in zip(corpus["titre"], corpus["texte"]):
    r = lisibilite(contenu, langue)
    toks = re.findall(r"\w+", contenu.lower())
    lignes.append({"texte": titre, "mots par phrase": r["mots par phrase"], "score": r["score"],
                   "niveau": interpreter(r["score"]), "TTR": round(len(set(toks)) / len(toks), 2)})
print(pd.DataFrame(lignes).to_string(index=False))

### Ce que vous devez voir
```
                                texte  mots par phrase  score          niveau  TTR
  Hugo, Discours sur la misère (1849)             26.0   71.5    assez facile 0.65
Jaurès, Discours à la jeunesse (1903)             48.0   50.7 assez difficile 0.71
```
Jaurès : une phrase de 48 mots, donc un score bas (50,7). Pourtant, à l'oral, l'anaphore « c'est… c'est… c'est… » rend ce passage facile à suivre.

**Observez :** un discours est fait pour être entendu. Quelles limites de la formule de lisibilité faudra-t-il discuter dans votre rapport ?

## À rendre
Répondez par écrit, en quelques lignes pour chaque question :
1. Le classement de la formule correspond-il au vôtre (début de la partie 1) ? Pour un texte où vous n'êtes pas d'accord, expliquez ce que la formule ne voit pas : vocabulaire rare, syntaxe, références culturelles, langue d'une autre époque.
2. Un collègue veut noter automatiquement les réponses de ses apprenants avec le taux de similarité. Écrivez-lui un avis argumenté (8 à 10 lignes), en vous appuyant sur les apprenants A, B et C.
3. Choisissez un texte pour une classe précise (niveau, objectif) et construisez une courte activité avec les outils du carnet : un score de lisibilité commenté, trois mots à expliquer, un texte à trous, deux questions de quiz. Indiquez ce que vous avez corrigé à la main dans ce que la machine a produit, et pourquoi.

## Ce qu'il faut retenir
- Une formule de lisibilité (Flesch, Kandel et Moles) ne mesure que la **longueur des phrases et des mots** : c'est un indice grossier, utile pour comparer des textes d'une même langue avec la même formule, jamais un verdict sur la difficulté réelle pour vos apprenants.
- Le **profil lexical** repose sur des comptages simples : occurrences, formes, TTR (à comparer à longueur égale), mots fréquents, hapax. Les chiffres signalent ; c'est la lecture qui interprète.
- La machine **fabrique vite** des exercices (textes à trous, quiz) et mesure des ressemblances ; l'enseignant garde le choix des mots, la qualité des distracteurs et le jugement sur le sens : **similarité n'est pas correction**.

Leçon correspondante sur le site du cours : [Lire et traiter des fichiers textes](https://progremer04.github.io/canva-au-for-ang-/#c2-fichiers) (occurrences et formes, mots vides, fréquences). Pour le mini-projet : [sujet 3, discours politiques](https://progremer04.github.io/canva-au-for-ang-/#mp-sujet-3).